# PrioritAI — Conformal Calibration Notebook
## Generating `q_hat` for the Stage2 FT ResNet50 Damage Classifier

---

### What this notebook does

This is an **offline calibration step**. It does not retrain or modify the model.

Using a labeled calibration dataset of **real-world images**, it computes `q_hat` —
the conformal threshold that gives a **coverage guarantee**: with `alpha = 0.15`,
the prediction set will contain the true damage class at least **85% of the time**
on data drawn from the same distribution.

### Architecture context

```
Frontend → Backend (FastAPI) → ML Service (FastAPI + Keras)
                                         ↑
                               reads q_hat.json at startup
```

### Model: Stage2 FT

This notebook is calibrated for **`ResNet50_3class_real_ft_s2_20260528_1753.keras`**.

The model was trained with `CLASS_NAMES = sorted(["heavy", "light", "medium"])`, so
class indices are assigned **alphabetically**:

| Index | Class  | damage_score |
|-------|--------|-------------|
| 0     | heavy  | 7           |
| 1     | light  | 3           |
| 2     | medium | 5           |

**⚠️ Do not change CLASS_NAMES ordering.** It must match the training config exactly.

### Calibration data

Calibrated on **real-world labeled images** (`labeled_to_test_list.txt` + `images_to_test/`).
This is the same domain as production traffic, giving more reliable coverage guarantees
than synthetic images.

### Output

`q_hat_stage2.json` saved to Google Drive — download and place at
`ml-service/model/q_hat.json` to activate in production.

---
## Step 1 — Mount Google Drive

All paths below are relative to your Drive root. Edit the `DRIVE_ROOT` variable in
Step 2 if your files are in a subfolder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Configuration

Edit these paths to match your Google Drive layout before running the rest of the notebook.

```
MyDrive/Project/
├── ResNet50_3class_real_ft_s2_20260528_1753.keras   ← MODEL_PATH
├── images_to_test/                                   ← IMAGES_DIR  (flat, no subfolders)
│   ├── 401.jpg
│   ├── 402.jpg
│   └── ...
├── labeled_to_test_list.txt                          ← LABEL_LIST_PATH
│   (format per line: "<image_id> <letter>"  h=heavy l=light m=medium)
└── conformal_output/
    └── q_hat_stage2.json                             ← OUTPUT_PATH (created automatically)
```

In [ ]:
# ── Google Drive paths ────────────────────────────────────────────────────────
DRIVE_ROOT       = "/content/drive/MyDrive"
MODEL_PATH       = f"{DRIVE_ROOT}/Project/ResNet50_3class_real_ft_s2_20260528_1753.keras"
IMAGES_DIR       = f"{DRIVE_ROOT}/Project/images_to_test"
LABEL_LIST_PATH  = f"{DRIVE_ROOT}/Project/labeled_to_test_list.txt"
OUTPUT_PATH      = f"{DRIVE_ROOT}/Project/conformal_output/q_hat_stage2.json"

# ── Must match production ml-service/app/inference.py ────────────────────────
# Stage2 FT was trained with sorted(["heavy","light","medium"]) — DO NOT reorder.
IMG_SIZE    = (224, 224)
CLASS_NAMES = ["heavy", "light", "medium"]   # index 0=heavy 1=light 2=medium

# ── Conformal calibration ─────────────────────────────────────────────────────
ALPHA       = 0.15    # target miscoverage rate → 85% coverage guarantee
BATCH_SIZE  = 32

# ── Label letter → class name mapping ────────────────────────────────────────
# labeled_to_test_list.txt uses single-letter codes (h/l/m)
LETTER_TO_CLASS = {"h": "heavy", "l": "light", "m": "medium"}

print("Configuration:")
print(f"  MODEL_PATH      : {MODEL_PATH}")
print(f"  IMAGES_DIR      : {IMAGES_DIR}")
print(f"  LABEL_LIST_PATH : {LABEL_LIST_PATH}")
print(f"  OUTPUT_PATH     : {OUTPUT_PATH}")
print(f"  IMG_SIZE        : {IMG_SIZE}")
print(f"  CLASS_NAMES     : {CLASS_NAMES}")
print(f"  ALPHA           : {ALPHA}")
print()
print("Class index mapping (must match training config):")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  index {idx} → {name}")

---
## Step 3 — Imports

All libraries are pre-installed in Google Colab. No `pip install` needed.

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pillow     : {Image.__version__}")

---
## Step 4 — Load the Keras Model

Loaded identically to production `preload_model()` in `ml-service/app/inference.py`.
A warmup predict runs immediately after loading to force graph compilation.

In [ ]:
print(f"Loading model from:\n  {MODEL_PATH}\n")

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}\n"
        "Check that MODEL_PATH in Step 2 points to the correct Drive location."
    )

model = tf.keras.models.load_model(MODEL_PATH)

# Warmup pass — identical to production preload_model()
dummy = np.zeros((1, *IMG_SIZE, 3), dtype=np.float32)
_ = model.predict(dummy, verbose=0)
print("Warmup complete.\n")

model.summary()

# Sanity: output layer must have exactly len(CLASS_NAMES) units
n_outputs = model.output_shape[-1]
if n_outputs != len(CLASS_NAMES):
    raise ValueError(
        f"Model output has {n_outputs} units but CLASS_NAMES has {len(CLASS_NAMES)} entries. "
        "Check that MODEL_PATH points to the correct Stage2 FT model."
    )
print(f"\nOutput units check: {n_outputs} == {len(CLASS_NAMES)}  ✓")

---
## Step 5 — Preprocessing Pipeline

Exact replica of `_preprocess()` in `ml-service/app/inference.py`.
Uses ResNet50's `preprocess_input` (caffe convention: subtract ImageNet channel means,
RGB→BGR). Input must be in **[0, 255]** — do NOT divide by 255 first.

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess_image(image_path: str) -> np.ndarray:
    """Preprocessing identical to ml-service/app/inference.py _preprocess()."""
    img = Image.open(image_path).convert("RGB")
    img = img.resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32)   # pixel values in [0, 255]
    return preprocess_input(arr)             # caffe convention: subtract means, RGB→BGR

print("preprocess_image() defined.")
print(f"Output shape per image: {IMG_SIZE + (3,)}")

---
## Step 6 — Load Calibration Dataset

Reads `labeled_to_test_list.txt`. Each line: `<image_id> <letter>`
where `h`=heavy, `l`=light, `m`=medium.

Images are loaded from the flat `IMAGES_DIR` directory.

Class label → index mapping matches the Stage2 FT training config:

| Letter | Class  | Index |
|--------|--------|-------|
| h      | heavy  | 0     |
| l      | light  | 1     |
| m      | medium | 2     |

In [ ]:
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths  = []   # List[Path]
image_labels = []   # List[int]
skipped      = []

images_dir  = Path(IMAGES_DIR)
label_file  = Path(LABEL_LIST_PATH)

if not label_file.is_file():
    raise FileNotFoundError(f"Label list not found: {label_file}")
if not images_dir.is_dir():
    raise FileNotFoundError(f"Images directory not found: {images_dir}")

print(f"Reading labels from : {label_file}")
print(f"Loading images from : {images_dir}")
print(f"CLASS_TO_IDX        : {CLASS_TO_IDX}")
print(f"LETTER_TO_CLASS     : {LETTER_TO_CLASS}")
print()

raw_lines = label_file.read_text(encoding="utf-8").splitlines()

for lineno, line in enumerate(raw_lines, start=1):
    line = line.strip()
    if not line or line.startswith("#"):
        continue

    parts = line.split()
    if len(parts) < 2:
        skipped.append(f"line {lineno}: too few fields — '{line}'")
        continue

    image_id = parts[0]
    letter   = parts[1].lower()

    if letter not in LETTER_TO_CLASS:
        skipped.append(f"line {lineno}: unknown letter '{letter}' — '{line}'")
        continue

    class_name = LETTER_TO_CLASS[letter]
    class_idx  = CLASS_TO_IDX[class_name]

    # Resolve image path: try as-is, then append common extensions
    candidate = images_dir / image_id
    if not candidate.exists():
        found = False
        for ext in VALID_EXTENSIONS:
            alt = images_dir / (image_id + ext)
            if alt.exists():
                candidate = alt
                found = True
                break
        if not found:
            skipped.append(f"line {lineno}: image not found — '{image_id}'")
            continue

    image_paths.append(candidate)
    image_labels.append(class_idx)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Loaded  : {len(image_paths)} images")
print(f"Skipped : {len(skipped)}")
if skipped:
    for msg in skipped[:10]:
        print(f"  [WARN] {msg}")
    if len(skipped) > 10:
        print(f"  ... and {len(skipped) - 10} more")

print("\nClass distribution (SANITY CHECK — verify these counts make sense):")
for class_name, idx in sorted(CLASS_TO_IDX.items(), key=lambda x: x[1]):
    count = sum(1 for lbl in image_labels if lbl == idx)
    print(f"  [{idx}] {class_name:8s} (letter={[k for k,v in LETTER_TO_CLASS.items() if v==class_name][0]}) → {count} images")
    if count == 0:
        print(f"       ⚠️  ZERO images for class '{class_name}' — calibration will be invalid!")

if len(image_paths) == 0:
    raise RuntimeError(
        "No images loaded. Check IMAGES_DIR and LABEL_LIST_PATH in Step 2."
    )

---
## Step 7 — Run Model Inference on All Calibration Images

Images are processed in batches of `BATCH_SIZE = 32`.

Output:
- `calibration_probs` — shape `(n, 3)`, raw softmax probabilities
- `calibration_labels` — shape `(n,)`, integer ground-truth class index

In [ ]:
n           = len(image_paths)
all_probs   = []
load_errors = 0

print(f"Running inference on {n} images (batch_size={BATCH_SIZE})...\n")
t0 = time.time()

for batch_start in range(0, n, BATCH_SIZE):
    batch_paths  = image_paths[batch_start : batch_start + BATCH_SIZE]
    batch_arrays = []

    for path in batch_paths:
        try:
            arr = preprocess_image(str(path))
            batch_arrays.append(arr)
        except Exception as exc:
            print(f"  [ERROR] {path.name}: {exc}")
            load_errors += 1
            batch_arrays.append(np.zeros((*IMG_SIZE, 3), dtype=np.float32))

    batch_tensor = np.stack(batch_arrays, axis=0)         # (B, 224, 224, 3)
    batch_probs  = model.predict(batch_tensor, verbose=0) # (B, 3)

    if batch_probs.shape[1] != len(CLASS_NAMES):
        raise ValueError(
            f"Model output shape mismatch: got {batch_probs.shape[1]} classes "
            f"but CLASS_NAMES has {len(CLASS_NAMES)}"
        )

    all_probs.append(batch_probs)

    done    = min(batch_start + BATCH_SIZE, n)
    elapsed = time.time() - t0
    pct     = done / n * 100
    print(f"  [{done:4d}/{n}]  {pct:5.1f}%   {elapsed:5.1f}s elapsed")

calibration_probs  = np.concatenate(all_probs, axis=0)       # (n, 3)
calibration_labels = np.array(image_labels, dtype=np.int32)  # (n,)

print(f"\nInference complete.")
print(f"  calibration_probs  shape : {calibration_probs.shape}")
print(f"  calibration_labels shape : {calibration_labels.shape}")

if load_errors > 0:
    print(f"\n[WARN] {load_errors} image(s) failed to load and were zero-filled.")

# Softmax sanity
row_sums = calibration_probs.sum(axis=1)
print(f"\nSoftmax row-sum check: min={row_sums.min():.4f}  max={row_sums.max():.4f}  (expect ~1.0)")

# Per-class mean confidence sanity
print("\nPer-class mean confidence (argmax column — sanity check):")
for class_name, idx in sorted(CLASS_TO_IDX.items(), key=lambda x: x[1]):
    mask = calibration_labels == idx
    if mask.sum() > 0:
        mean_conf = calibration_probs[mask, idx].mean()
        print(f"  [{idx}] {class_name:8s} → mean p[{idx}] = {mean_conf:.4f}")
    else:
        print(f"  [{idx}] {class_name:8s} → no samples")

---
## Step 8 — Conformal Calibration Algorithm

### Theory

**APS (Adaptive Prediction Sets)** — more robust than standard conformal for
overconfident classifiers.

**Nonconformity score** for image `i`:
$$s_i = \text{cumulative probability from top-1 down to true class}$$

A low score means the model ranked the true class near the top (confident and correct).
A high score (→1.0) means the true class was buried.

**Calibration quantile:**
$$\hat{q} = \text{Quantile}\!\left(s_1, \ldots, s_n,\ \frac{\lceil (n+1)(1-\alpha) \rceil}{n}\right)$$

**At runtime**, prediction set for a new image accumulates classes by descending
probability until the cumulative sum ≥ q_hat.

### Implementation

This is the **exact algorithm** ported to `ml-service/app/inference.py`. Do not modify it.

In [ ]:
def compute_conformal_threshold(calibration_probs, calibration_labels, alpha=0.15):
    n = len(calibration_probs)
    if n == 0:
        return 1.0

    scores = []
    for probs, true_label in zip(calibration_probs, calibration_labels):
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs   = probs[sorted_indices]
        true_rank      = np.where(sorted_indices == true_label)[0][0]
        cumulative_prob = np.sum(sorted_probs[:true_rank + 1])
        scores.append(cumulative_prob)

    scores  = np.array(scores)
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)
    q_hat   = np.quantile(scores, q_level, method="higher")
    return q_hat


def predict_conformal(model_prediction, q_hat, class_names):
    """Exact port of predict_conformal_aps() in ml-service/app/inference.py."""
    sorted_indices  = np.argsort(model_prediction)[::-1]
    prediction_set  = []
    cumulative_prob = 0.0
    for idx in sorted_indices:
        prediction_set.append(class_names[idx])
        cumulative_prob += model_prediction[idx]
        if cumulative_prob >= q_hat:
            break
    return prediction_set


print("APS conformal functions defined.")

---
## Step 9 — Compute `q_hat`

In [ ]:
q_hat = compute_conformal_threshold(calibration_probs, calibration_labels, alpha=ALPHA)

print("Conformal calibration complete")
print(f"  n (calibration images) : {len(calibration_probs)}")
print(f"  alpha                  : {ALPHA}  (target miscoverage rate)")
print(f"  coverage guarantee     : {(1 - ALPHA) * 100:.0f}%")
print(f"  q_hat                  : {q_hat:.6f}")
print()
print("Interpretation: prediction set accumulates classes by descending probability")
print(f"until cumulative sum >= {q_hat:.4f}")

In [ ]:
print("Sample APS prediction sets (first 20 calibration images):\n")
for i in range(min(20, len(calibration_probs))):
    probs      = calibration_probs[i]
    pred_set   = predict_conformal(probs, q_hat, CLASS_NAMES)
    pred_class = CLASS_NAMES[np.argmax(probs)]
    true_class = CLASS_NAMES[calibration_labels[i]]
    covered    = true_class in pred_set
    marker     = "OK" if covered else "MISS"
    print(
        f"  [{marker:4s}]  true={true_class:8s} | "
        f"pred={pred_class:8s} | "
        f"probs={[round(float(p), 3) for p in probs]} | "
        f"set={pred_set}"
    )

---
## Step 10 — Save `q_hat_stage2.json`

In [ ]:
output_dir = Path(OUTPUT_PATH).parent
output_dir.mkdir(parents=True, exist_ok=True)

payload = {
    "q_hat":          float(q_hat),
    "alpha":          ALPHA,
    "n_calibration":  int(len(calibration_probs)),
    "class_names":    CLASS_NAMES,
    "model_path":     MODEL_PATH,
    "date_saved":     datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Saved → {OUTPUT_PATH}\n")
print(json.dumps(payload, indent=2))

---
## Step 11 — Sanity Checks

1. **Sample predictions** — 10 random calibration images; true class must be in set.
2. **Full calibration set coverage** — must be ≥ `1 - alpha` (85%).
3. **Prediction set size distribution** — shows model uncertainty profile.

In [ ]:
print("=" * 65)
print("SANITY CHECK 1 — prediction sets on 10 random calibration images")
print("=" * 65)

rng     = np.random.default_rng(42)
indices = rng.choice(len(calibration_probs), size=min(10, len(calibration_probs)), replace=False)

covered_sample = 0
for i in indices:
    probs      = calibration_probs[i]
    true_label = CLASS_NAMES[calibration_labels[i]]
    pred_set   = predict_conformal(probs, q_hat, CLASS_NAMES)
    is_covered = true_label in pred_set
    covered_sample += int(is_covered)
    marker = "OK" if is_covered else "MISS"
    print(
        f"  [{marker:4s}]  true={true_label:8s}  "
        f"probs={[round(float(p), 3) for p in probs]}  "
        f"set={pred_set}"
    )

print(f"\nSample coverage: {covered_sample}/{len(indices)} = {covered_sample / len(indices):.0%}")

# ── Full calibration set coverage ─────────────────────────────────────────────
print()
print("=" * 65)
print("SANITY CHECK 2 — coverage over full calibration set")
print("=" * 65)

covered_full = sum(
    CLASS_NAMES[calibration_labels[i]] in predict_conformal(
        calibration_probs[i], q_hat, CLASS_NAMES
    )
    for i in range(len(calibration_probs))
)
full_coverage = covered_full / len(calibration_probs)
status = "PASS" if full_coverage >= (1 - ALPHA) else "FAIL"
print(f"  Coverage : {full_coverage:.4f}  (target >= {1 - ALPHA:.2f})  [{status}]")
if status == "FAIL":
    print("  ⚠️  Coverage below target. Check CLASS_NAMES order and label file correctness.")

# ── Set size distribution ─────────────────────────────────────────────────────
print()
print("=" * 65)
print("SANITY CHECK 3 — prediction set size distribution")
print("=" * 65)

set_sizes   = [
    len(predict_conformal(calibration_probs[i], q_hat, CLASS_NAMES))
    for i in range(len(calibration_probs))
]
size_counts = {s: set_sizes.count(s) for s in sorted(set(set_sizes))}

for size, count in size_counts.items():
    bar = "#" * int(count / len(set_sizes) * 40)
    print(f"  size {size}: {count:5d} images ({count / len(set_sizes):.1%})  {bar}")

print(f"\n  Mean set size : {np.mean(set_sizes):.3f}")
print(f"  (1.0 = model always certain, {len(CLASS_NAMES)}.0 = model always uncertain)")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"  model          : {Path(MODEL_PATH).name}")
print(f"  class_names    : {CLASS_NAMES}")
print(f"  q_hat          : {q_hat:.6f}")
print(f"  alpha          : {ALPHA}")
print(f"  n_calibration  : {len(calibration_probs)}")
print(f"  coverage       : {full_coverage:.4f}  [{status}]")
print(f"  mean set size  : {np.mean(set_sizes):.3f}")
print(f"  output file    : {OUTPUT_PATH}")

---
## Next Steps

### Deploy the new `q_hat_stage2.json`

1. **Download** `q_hat_stage2.json` from Google Drive.
2. **Replace** `ml-service/model/q_hat.json` with it.
3. **Rebuild and restart** the ML service:
   ```bash
   docker compose build ml-service
   docker compose up -d ml-service
   ```
4. **Check startup logs** — you should see:
   ```
   Loaded APS threshold: q_hat=<value> (alpha=0.15, n_calibration=<n>)
   ```
   If you see a `class_names mismatch` warning, the wrong q_hat file is loaded.

### Expected `q_hat_stage2.json` structure

```json
{
  "q_hat": <float>,
  "alpha": 0.15,
  "n_calibration": <int>,
  "class_names": ["heavy", "light", "medium"],
  "model_path": "<Drive path to Stage2 FT model>",
  "date_saved": "<ISO 8601>"
}
```

### Important notes

- `class_names` must be `["heavy", "light", "medium"]` (alphabetical, matching Stage2 FT training).
- `q_hat` is specific to the `alpha` value used. If you change `alpha`, re-run the notebook.
- Re-run calibration whenever the model is updated.